In [4]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import nflreadpy as nfl

In [5]:
MIN_GAMES = 20
START_YEAR = 2024
END_YEAR = 2026
DEF_POSITIONS = {'DB', 'LB', 'DL'}
OF_POSITIONS = {'OL', 'WR', 'RB', 'TE'}
BIO_COLS = ['player_id', 'first_name', 'last_name', 'position', 'total_teams', 'current_team', 'games', 'seasons']
QB_COLS = ['completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'passing_epa',
           'carries', 'rushing_yards', 'rushing_tds', 'rushing_epa']
OF_COLS = ['carries', 'rushing_yards', 'rushing_tds', 'rushing_epa', 'receptions', 'targets',
           'receiving_yards', 'receiving_tds', 'receiving_epa']
DEF_COLS = ['def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_sacks', 'def_sack_yards',
            'def_qb_hits', 'def_interceptions']
ST_COLS = ['fg_made', 'fg_att', 'fg_long', 'fg_made_distance', 
           'pt_att', 'pt_inside_20', 'pt_downed', 'pt_touchback', 'pt_fair_caught', 'pt_yards']

raw = nfl.load_player_stats(seasons = list(range(START_YEAR, END_YEAR + 1)), summary_level = 'reg').to_pandas()
list(raw.columns)

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'season_type',
 'recent_team',
 'games',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'receiv

In [6]:
raw.position_group.value_counts()

position_group
DB      1193
LB       900
DL       899
OL       747
WR       710
RB       469
TE       392
SPEC     312
QB       230
Name: count, dtype: int64

Not all players have two word names (ex. Jordan van den Berg). I will make first name the first word and last name the rest of the words. 

In [7]:
def get_player_stats(data, position, cols):
    data = data.copy()

    positions = {position} if isinstance(position, str) else position

    stat_aggs = {col: (col, 'max' if col == 'fg_long' else 'sum') for col in cols}

    player_stats = (
        data
        .sort_values(by = 'season', ascending = True)
        .loc[data['position_group'].isin(positions)]
        .groupby('player_id', as_index = False)
        .agg(
            name = ('player_display_name', 'last'),
            position = ('position_group', 'last'),
            total_teams = ('recent_team', 'nunique'),
            current_team = ('recent_team', 'last'),
            games = ('games', 'sum'),
            seasons = ('games', 'count'),
            **stat_aggs
        )
    )

    player_stats['first_name'] = [name.split()[0] for name in player_stats['name']]
    player_stats['last_name'] = [" ".join(name.split()[1:]) for name in player_stats['name']]

    return player_stats[BIO_COLS + cols].loc[player_stats.games >= MIN_GAMES].reset_index(drop = True)


In [8]:
qb_stats = get_player_stats(raw, 'QB', QB_COLS)
of_stats = get_player_stats(raw, OF_POSITIONS, OF_COLS)
def_stats = get_player_stats(raw, DEF_POSITIONS, DEF_COLS)
st_stats = get_player_stats(raw, 'SPEC', ST_COLS)

In [9]:
qb_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,completions,attempts,passing_yards,passing_tds,passing_interceptions,passing_epa,carries,rushing_yards,rushing_tds,rushing_epa
0,00-0023459,Aaron,Rodgers,QB,2,PIT,43,3,915,1443,9460,68,24,17.608749,62,219,1,-1.114456
1,00-0026158,Joe,Flacco,QB,2,CIN,20,2,413,664,4240,27,17,-38.819739,30,61,1,-1.578302
2,00-0026498,Matthew,Stafford,QB,1,LA,43,3,976,1486,11092,85,26,230.363204,76,59,0,-47.913456
3,00-0029604,Kirk,Cousins,QB,2,MIN,35,3,718,1106,8011,50,31,71.697772,63,25,3,-24.450609
4,00-0030565,Geno,Smith,QB,3,NYJ,43,3,922,1339,9747,55,35,-25.834289,119,439,2,-10.369054


In [10]:
of_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,carries,rushing_yards,rushing_tds,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_epa
0,00-0029892,Kyle,Juszczyk,RB,1,SF,37,3,8,27,1,-10.463872,53,72,502,6,12.021536
1,00-0030035,Adam,Thielen,WR,2,PIT,24,2,0,0,0,0.000000,67,97,801,5,45.608586
2,00-0030061,Zach,Ertz,TE,2,PHI,33,3,0,0,0,0.000000,119,171,1180,11,39.100553
3,00-0030279,Keenan,Allen,WR,3,SF,42,3,0,0,0,0.000000,199,312,1971,12,67.590428
4,00-0030506,Travis,Kelce,TE,1,KC,43,3,3,3,1,-0.198870,214,298,2260,12,90.833770


In [11]:
def_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_sacks,def_sack_yards,def_qb_hits,def_interceptions
0,00-0026190,Calais,Campbell,DL,3,BAL,44,3,54,10,48,12.5,98.0,36,0
1,00-0027940,Von,Miller,LB,3,DAL,31,3,31,2,16,16.0,130.0,24,0
2,00-0027962,Cameron,Jordan,LB,1,NO,39,3,52,5,39,14.5,90.5,24,1
3,00-0027969,Cameron,Heyward,DL,1,PIT,44,3,80,8,95,14.0,88.5,37,0
4,00-0029255,Bobby,Wagner,LB,1,WAS,34,2,141,13,140,6.5,35.0,17,2


In [12]:
st_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,fg_made,fg_att,fg_long,fg_made_distance,pt_att,pt_inside_20,pt_downed,pt_touchback,pt_fair_caught,pt_yards
0,00-0025565,Nick,Folk,SPEC,3,ATL,39,3,62,69,58.0,2459,0,0,0,0,0,0
1,00-0027114,Thomas,Morstead,SPEC,3,TEN,34,3,0,0,NaN,0,114,44,11,8,33,5217
2,00-0028872,Johnny,Hekker,SPEC,3,MIN,36,3,0,0,NaN,0,163,48,27,11,45,7451
3,00-0029692,Bryan,Anger,SPEC,1,DAL,44,3,0,0,NaN,0,129,52,8,4,44,6212
4,00-0029822,Brandon,McManus,SPEC,2,LA,29,3,49,56,56.0,1697,0,0,0,0,0,0


In [13]:
player_list = (
    pd.concat([qb_stats, of_stats, def_stats, st_stats], ignore_index = True)[BIO_COLS]
    .drop_duplicates(subset = 'player_id')
    .reset_index(drop = True)
)

player_list.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons
0,00-0023459,Aaron,Rodgers,QB,2,PIT,43,3
1,00-0026158,Joe,Flacco,QB,2,CIN,20,2
2,00-0026498,Matthew,Stafford,QB,1,LA,43,3
3,00-0029604,Kirk,Cousins,QB,2,MIN,35,3
4,00-0030565,Geno,Smith,QB,3,NYJ,43,3


In [14]:
player_list.to_csv('data/player_list.csv', index = False)
stats = {'qb': qb_stats, 'of': of_stats, 'def': def_stats, 'st': st_stats}
for pos, df in stats.items():
    df.to_csv(f'data/{pos}_stats.csv', index = False)

## Player Evaluation